# Phase 1: SingBERT Fine-Tuning Experiments

**Notebook:** `notebooks/SingBERT/singbert_sg_experiments.ipynb`

## Experiments

| Exp | Description | Training data | Stages |
|-----|-------------|---------------|--------|
| A | SingBERT → SG only | SG synthetic train | 1 |
| B | SingBERT → HateXplain → SG | HateXplain train, then SG train | 2 |
| C | SingBERT + Combined | HateXplain + SG oversampled (merged) | 1 |

## Evaluation
Each experiment is evaluated on two test sets:
- **SG test set** — `data/synthetic/20_terms/split_60_20_20/test.jsonl`
- **HateXplain test set** — `data/hatexplain/hatexplain_test.jsonl`

## Label mapping
Both datasets use the same 3 classes but different conventions — unified here:

| ID | Label | HateXplain annotator vote | SG jsonl label |
|----|-------|--------------------------|----------------|
| 0 | hate | 0 | `hate` |
| 1 | normal | 1 | `normal` |
| 2 | offensive | 2 | `offensive` |

---

## Cell 1 — Install Dependencies

In [1]:
!pip install transformers torch scikit-learn pandas tqdm --quiet

## Cell 2 — Config

All paths and hyperparameters are defined here. Adjust before running.

In [2]:
import os

# ── Paths (relative to repo root / this notebook's working directory) ─────────
# Adjust BASE_DIR if running from a different working directory
BASE_DIR = "../.."  # notebook is at notebooks/SingBERT/

SG_TRAIN_PATH  = os.path.join(BASE_DIR, "data/singlish/synthetic/20_terms/split_60_20_20/train.jsonl")
SG_VAL_PATH    = os.path.join(BASE_DIR, "data/singlish/synthetic/20_terms/split_60_20_20/val.jsonl")
SG_TEST_PATH   = os.path.join(BASE_DIR, "data/singlish/synthetic/20_terms/split_60_20_20/test.jsonl")
HX_TRAIN_PATH  = os.path.join(BASE_DIR, "data/hatexplain/hatexplain_train.jsonl")
HX_TEST_PATH   = os.path.join(BASE_DIR, "data/hatexplain/hatexplain_test.jsonl")

# ── Model ──────────────────────────────────────────────────────────────────────
SINGBERT_MODEL_NAME = "zanelim/singbert"

# ── Output directories for saved checkpoints ──────────────────────────────────
OUT_EXP_A          = os.path.join(BASE_DIR, "models/singbert_exp_a")
OUT_EXP_B_STAGE1   = os.path.join(BASE_DIR, "models/singbert_exp_b_stage1")  # after HateXplain
OUT_EXP_B_STAGE2   = os.path.join(BASE_DIR, "models/singbert_exp_b_stage2")  # after SG
OUT_EXP_C          = os.path.join(BASE_DIR, "models/singbert_exp_c")

# ── Training hyperparameters ───────────────────────────────────────────────────
MAX_LENGTH      = 512
BATCH_SIZE      = 16
LEARNING_RATE   = 2e-5
EPOCHS_SG       = 5    # SG dataset is small — more epochs needed
EPOCHS_HX       = 3    # HateXplain is large — fewer epochs sufficient
EPOCHS_COMBINED = 4

# ── Exp C: SG oversample ratio ─────────────────────────────────────────────────
# SG rows will be repeated until they make up this fraction of the combined set
SG_OVERSAMPLE_RATIO = 0.10  # 10%

# ── Label mapping ──────────────────────────────────────────────────────────────
# HateXplain annotator votes: 0=hate, 1=normal, 2=offensive
# SG dataset label strings: 'hate', 'normal', 'offensive'
# Unified ID scheme used throughout this notebook:
ID2LABEL  = {0: "hate", 1: "normal", 2: "offensive"}
LABEL2ID  = {"hate": 0, "normal": 1, "offensive": 2}
LABEL_NAMES = ["hate", "normal", "offensive"]

# Verify paths exist
for name, path in [
    ("SG train", SG_TRAIN_PATH), ("SG val", SG_VAL_PATH), ("SG test", SG_TEST_PATH),
    ("HX train", HX_TRAIN_PATH), ("HX test", HX_TEST_PATH)
]:
    status = "✅" if os.path.exists(path) else "❌ NOT FOUND"
    print(f"  {status}  {name}: {path}")

  ✅  SG train: ../..\data/singlish/synthetic/20_terms/split_60_20_20/train.jsonl
  ✅  SG val: ../..\data/singlish/synthetic/20_terms/split_60_20_20/val.jsonl
  ✅  SG test: ../..\data/singlish/synthetic/20_terms/split_60_20_20/test.jsonl
  ✅  HX train: ../..\data/hatexplain/hatexplain_train.jsonl
  ✅  HX test: ../..\data/hatexplain/hatexplain_test.jsonl


## Cell 3 — Data Loading & Preprocessing

### HateXplain format notes
- `post_tokens` — list of word tokens; joined with spaces to form the sentence
- `annotators.label` — list of votes from multiple annotators (`0=hate, 1=normal, 2=offensive`)
- Final label = **majority vote**; ties broken by taking the lower ID (more severe label wins)

### SG format notes
- `text` — sentence string
- `label` — string: `hate`, `normal`, or `offensive`

In [3]:
import json
import pandas as pd
from collections import Counter


def majority_vote(votes):
    """
    Return the majority label from a list of annotator votes.
    Ties broken by lowest ID (more severe label wins: hate < normal < offensive).
    """
    counts = Counter(votes)
    max_count = max(counts.values())
    candidates = [label for label, count in counts.items() if count == max_count]
    return min(candidates)  # lowest ID wins on tie


def load_hatexplain(path):
    """
    Load a HateXplain jsonl file.
    Reconstructs text from post_tokens and derives label via majority vote.
    Returns a DataFrame with columns: text, label_id, label
    """
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            d = json.loads(line)
            text = " ".join(d["post_tokens"])
            votes = d["annotators"]["label"]
            label_id = majority_vote(votes)
            records.append({
                "text":     text,
                "label_id": label_id,
                "label":    ID2LABEL[label_id]
            })
    return pd.DataFrame(records)


def load_sg(path):
    """
    Load a SG synthetic jsonl file.
    Returns a DataFrame with columns: text, label_id, label
    """
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            d = json.loads(line)
            label = d["label"]
            records.append({
                "text":     d["text"],
                "label_id": LABEL2ID[label],
                "label":    label
            })
    return pd.DataFrame(records)


# Load all splits
hx_train_df = load_hatexplain(HX_TRAIN_PATH)
hx_test_df  = load_hatexplain(HX_TEST_PATH)
sg_train_df = load_sg(SG_TRAIN_PATH)
sg_val_df   = load_sg(SG_VAL_PATH)
sg_test_df  = load_sg(SG_TEST_PATH)

print("Dataset sizes:")
print(f"  HateXplain train : {len(hx_train_df):>5} rows  | {dict(hx_train_df['label'].value_counts())}")
print(f"  HateXplain test  : {len(hx_test_df):>5} rows  | {dict(hx_test_df['label'].value_counts())}")
print(f"  SG train         : {len(sg_train_df):>5} rows  | {dict(sg_train_df['label'].value_counts())}")
print(f"  SG val           : {len(sg_val_df):>5} rows  | {dict(sg_val_df['label'].value_counts())}")
print(f"  SG test          : {len(sg_test_df):>5} rows  | {dict(sg_test_df['label'].value_counts())}")

c:\Users\keith\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Dataset sizes:
  HateXplain train : 15383 rows  | {'normal': 6251, 'hate': 4748, 'offensive': 4384}
  HateXplain test  :  1924 rows  | {'normal': 782, 'hate': 594, 'offensive': 548}
  SG train         :   319 rows  | {'normal': 112, 'hate': 108, 'offensive': 99}
  SG val           :   106 rows  | {'normal': 37, 'hate': 36, 'offensive': 33}
  SG test          :   108 rows  | {'normal': 38, 'hate': 37, 'offensive': 33}


## Cell 4 — Dataset Class & Training Utilities

In [6]:
import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from sklearn.metrics import classification_report, accuracy_score, f1_score


class HateSpeechDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=MAX_LENGTH):
        self.texts     = dataframe["text"].tolist()
        self.labels    = dataframe["label_id"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long)
        }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0)
    }


def make_training_args(output_dir, epochs, lr=LEARNING_RATE, batch_size=BATCH_SIZE):
    """Return a TrainingArguments object with consistent settings across experiments."""
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_dir=os.path.join(output_dir, "logs"),
        logging_steps=50,
        save_total_limit=2,       # keep only best + latest checkpoint
        report_to="none",         # disable wandb/tensorboard unless you want it
        fp16=torch.cuda.is_available(),  # use mixed precision if GPU available
    )


def load_fresh_singbert():
    """Load a fresh SingBERT with a randomly initialised 3-class head."""
    tokenizer = BertTokenizer.from_pretrained(SINGBERT_MODEL_NAME)
    model = BertForSequenceClassification.from_pretrained(
        SINGBERT_MODEL_NAME,
        num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True
    )
    return tokenizer, model


def evaluate_on(model, tokenizer, dataframe, split_name):
    """
    Evaluate a model on a given dataframe split.
    Prints accuracy, macro F1, and per-class classification report.
    Returns (accuracy, macro_f1).
    """
    model.eval()
    dataset = HateSpeechDataset(dataframe, tokenizer)
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for i in range(len(dataset)):
            item = dataset[i]
            input_ids      = item["input_ids"].unsqueeze(0)
            attention_mask = item["attention_mask"].unsqueeze(0)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            pred = torch.argmax(logits, dim=1).item()
            all_preds.append(pred)
            all_labels.append(item["labels"].item())

    acc      = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    print(f"\n── {split_name}")
    print(f"   Accuracy : {acc:.4f}")
    print(f"   Macro F1 : {macro_f1:.4f}")
    print(classification_report(
        all_labels, all_preds,
        labels=[0, 1, 2],
        target_names=LABEL_NAMES,
        zero_division=0
    ))
    return acc, macro_f1


print("✅ Dataset class and training utilities ready")
print(f"   Device: {'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")

✅ Dataset class and training utilities ready
   Device: CPU


## Cell 5 — Experiment A: SingBERT → SG Only

Fine-tune SingBERT directly on the SG synthetic training data. This is the simplest baseline — no prior hate speech knowledge, just Singlish language understanding.

In [7]:
print("="*60)
print("EXP A: SingBERT → SG only")
print("="*60)

# Load fresh model
tokenizer_a, model_a = load_fresh_singbert()

# Datasets
train_ds_a = HateSpeechDataset(sg_train_df, tokenizer_a)
val_ds_a   = HateSpeechDataset(sg_val_df,   tokenizer_a)

# Train
trainer_a = Trainer(
    model=model_a,
    args=make_training_args(OUT_EXP_A, epochs=EPOCHS_SG),
    train_dataset=train_ds_a,
    eval_dataset=val_ds_a,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
trainer_a.train()

# Save
model_a.save_pretrained(OUT_EXP_A)
tokenizer_a.save_pretrained(OUT_EXP_A)
print(f"\n✅ Exp A model saved to {OUT_EXP_A}")

# Evaluate on both test sets
print("\nExp A — Evaluation:")
exp_a_sg_acc, exp_a_sg_f1 = evaluate_on(model_a, tokenizer_a, sg_test_df,  "SG test set")
exp_a_hx_acc, exp_a_hx_f1 = evaluate_on(model_a, tokenizer_a, hx_test_df, "HateXplain test set")

EXP A: SingBERT → SG only


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at zanelim/singbert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\keith\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Cell 6 — Experiment B: SingBERT → HateXplain → SG (Two-Stage)

**Stage 1:** Fine-tune SingBERT on HateXplain — teaches general hate speech patterns in English.

**Stage 2:** Continue fine-tuning on SG synthetic data — adapts to Singlish-specific patterns.

The Stage 1 checkpoint is saved separately so you can measure catastrophic forgetting (how much HateXplain performance drops after Stage 2).

In [ ]:
print("="*60)
print("EXP B — Stage 1: SingBERT → HateXplain")
print("="*60)

# Load fresh model for Stage 1
tokenizer_b, model_b = load_fresh_singbert()

# HateXplain uses the same tokenizer — just wrap in dataset class
# We use SG val set for early stopping here since we care about
# downstream SG performance, not HateXplain val performance
hx_train_ds = HateSpeechDataset(hx_train_df, tokenizer_b)
sg_val_ds_b = HateSpeechDataset(sg_val_df,   tokenizer_b)

trainer_b1 = Trainer(
    model=model_b,
    args=make_training_args(OUT_EXP_B_STAGE1, epochs=EPOCHS_HX),
    train_dataset=hx_train_ds,
    eval_dataset=sg_val_ds_b,
    compute_metrics=compute_metrics,
)
trainer_b1.train()

# Save Stage 1 checkpoint — important for forgetting analysis
model_b.save_pretrained(OUT_EXP_B_STAGE1)
tokenizer_b.save_pretrained(OUT_EXP_B_STAGE1)
print(f"\n✅ Exp B Stage 1 checkpoint saved to {OUT_EXP_B_STAGE1}")

# Evaluate Stage 1 on both test sets
print("\nExp B Stage 1 — Evaluation (before SG fine-tuning):")
exp_b1_sg_acc, exp_b1_sg_f1 = evaluate_on(model_b, tokenizer_b, sg_test_df,  "SG test set")
exp_b1_hx_acc, exp_b1_hx_f1 = evaluate_on(model_b, tokenizer_b, hx_test_df, "HateXplain test set")

In [ ]:
print("="*60)
print("EXP B — Stage 2: SingBERT (post-HateXplain) → SG")
print("="*60)

# Continue from Stage 1 weights (model_b already holds them)
sg_train_ds_b = HateSpeechDataset(sg_train_df, tokenizer_b)

trainer_b2 = Trainer(
    model=model_b,
    args=make_training_args(OUT_EXP_B_STAGE2, epochs=EPOCHS_SG),
    train_dataset=sg_train_ds_b,
    eval_dataset=sg_val_ds_b,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
trainer_b2.train()

# Save Stage 2 checkpoint
model_b.save_pretrained(OUT_EXP_B_STAGE2)
tokenizer_b.save_pretrained(OUT_EXP_B_STAGE2)
print(f"\n✅ Exp B Stage 2 model saved to {OUT_EXP_B_STAGE2}")

# Evaluate Stage 2 on both test sets
print("\nExp B Stage 2 — Evaluation (after SG fine-tuning):")
exp_b2_sg_acc, exp_b2_sg_f1 = evaluate_on(model_b, tokenizer_b, sg_test_df,  "SG test set")
exp_b2_hx_acc, exp_b2_hx_f1 = evaluate_on(model_b, tokenizer_b, hx_test_df, "HateXplain test set")

# Forgetting analysis
print("\n── Catastrophic Forgetting Check (HateXplain performance):")
print(f"   Stage 1 HateXplain Macro F1 : {exp_b1_hx_f1:.4f}")
print(f"   Stage 2 HateXplain Macro F1 : {exp_b2_hx_f1:.4f}")
delta = exp_b2_hx_f1 - exp_b1_hx_f1
print(f"   Delta                        : {delta:+.4f} {'(forgetting detected)' if delta < -0.05 else '(acceptable)'}")

## Cell 7 — Experiment C: SingBERT + Combined Training

Merge HateXplain and SG training data in a single training run. SG rows are oversampled so they make up `SG_OVERSAMPLE_RATIO` (default 10%) of the combined set — prevents the much larger HateXplain set from drowning out SG signal.

In [ ]:
import math

print("="*60)
print("EXP C: SingBERT + Combined (HateXplain + SG oversampled)")
print("="*60)

# ── Build combined dataset ─────────────────────────────────────────────────────
# Target: SG rows = SG_OVERSAMPLE_RATIO of total
# So if HateXplain has N rows and we want SG to be 10%:
#   sg_copies = ceil((N * SG_OVERSAMPLE_RATIO) / (1 - SG_OVERSAMPLE_RATIO) / len(sg_train_df))
n_hx   = len(hx_train_df)
n_sg   = len(sg_train_df)
target_sg_count = math.ceil((n_hx * SG_OVERSAMPLE_RATIO) / (1 - SG_OVERSAMPLE_RATIO))
sg_copies = math.ceil(target_sg_count / n_sg)
sg_oversampled = pd.concat([sg_train_df] * sg_copies, ignore_index=True)

combined_df = pd.concat([hx_train_df, sg_oversampled], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

actual_sg_pct = len(sg_oversampled) / len(combined_df) * 100
print(f"  HateXplain rows  : {n_hx}")
print(f"  SG rows (orig)   : {n_sg}")
print(f"  SG copies        : {sg_copies}x → {len(sg_oversampled)} rows")
print(f"  Combined total   : {len(combined_df)} rows")
print(f"  SG % of combined : {actual_sg_pct:.1f}% (target: {SG_OVERSAMPLE_RATIO*100:.0f}%)")
print(f"  Label distribution: {dict(combined_df['label'].value_counts())}")

# ── Train ──────────────────────────────────────────────────────────────────────
tokenizer_c, model_c = load_fresh_singbert()

train_ds_c = HateSpeechDataset(combined_df, tokenizer_c)
val_ds_c   = HateSpeechDataset(sg_val_df,   tokenizer_c)  # validate on SG

trainer_c = Trainer(
    model=model_c,
    args=make_training_args(OUT_EXP_C, epochs=EPOCHS_COMBINED),
    train_dataset=train_ds_c,
    eval_dataset=val_ds_c,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
trainer_c.train()

# Save
model_c.save_pretrained(OUT_EXP_C)
tokenizer_c.save_pretrained(OUT_EXP_C)
print(f"\n✅ Exp C model saved to {OUT_EXP_C}")

# Evaluate on both test sets
print("\nExp C — Evaluation:")
exp_c_sg_acc, exp_c_sg_f1 = evaluate_on(model_c, tokenizer_c, sg_test_df,  "SG test set")
exp_c_hx_acc, exp_c_hx_f1 = evaluate_on(model_c, tokenizer_c, hx_test_df, "HateXplain test set")

## Cell 8 — Results Summary

Consolidates all experiment results into a comparison table. Run this after all experiments complete.

In [ ]:
print("="*70)
print("PHASE 1 RESULTS SUMMARY")
print("="*70)

results = [
    ("Exp A  — SingBERT → SG",              exp_a_sg_f1,  exp_a_hx_f1),
    ("Exp B1 — SingBERT → HateXplain",      exp_b1_sg_f1, exp_b1_hx_f1),
    ("Exp B2 — SingBERT → HateXplain → SG", exp_b2_sg_f1, exp_b2_hx_f1),
    ("Exp C  — SingBERT + Combined",         exp_c_sg_f1,  exp_c_hx_f1),
]

print(f"\n{'Experiment':<42} {'SG Macro F1':>12} {'HX Macro F1':>12}")
print("-" * 68)
for name, sg_f1, hx_f1 in results:
    print(f"{name:<42} {sg_f1:>12.4f} {hx_f1:>12.4f}")

print("\n" + "="*70)
print("FORGETTING ANALYSIS (Exp B — HateXplain knowledge retained after SG)")
print("="*70)
print(f"  HateXplain F1 after Stage 1 : {exp_b1_hx_f1:.4f}")
print(f"  HateXplain F1 after Stage 2 : {exp_b2_hx_f1:.4f}")
print(f"  Drop                        : {exp_b2_hx_f1 - exp_b1_hx_f1:+.4f}")

print("\n" + "="*70)
print("INTERPRETATION GUIDE")
print("="*70)
print("""
  SG Macro F1  — how well the model handles Singlish hate speech
  HX Macro F1  — how well the model retains general English hate speech knowledge

  Exp A high SG, low HX  → SG data alone is sufficient but not generalisable
  Exp B2 high both       → Two-stage training transfers well without forgetting
  Exp C high both        → Combined training achieves best of both worlds
  Exp B2 high SG, low HX → Catastrophic forgetting — SG fine-tuning overrides HateXplain
""")